<a href="https://colab.research.google.com/github/marwa-nassane0052/quantum_ai_experiment/blob/main/test6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

folder_path = '/content/drive/MyDrive/skin_cancer_processed_dataset_data_augmentation'

if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Folder '{folder_path}' not found. Please check the path and ensure it's in your My Drive.")

Contents of '/content/drive/MyDrive/skin_cancer_processed_dataset_data_augmentation':
y_train.npy
y_test.npy
X_train.npy
X_test.npy


In [3]:
import numpy as np
folder_path = '/content/drive/MyDrive/skin_cancer_processed_dataset_data_augmentation'
X_train = np.load(os.path.join(folder_path, 'X_train.npy'))
y_train = np.load(os.path.join(folder_path, 'y_train.npy'))
X_test = np.load(os.path.join(folder_path, 'X_test.npy'))
y_test = np.load(os.path.join(folder_path, 'y_test.npy'))

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (2200, 3, 128, 128)
y_train shape: (2200,)
X_test shape: (660, 3, 128, 128)
y_test shape: (660,)


**Bulding the hybrid model**

In [4]:
!pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 20.5 MB/s eta 0:00:00


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# ── Convert dataset to tensors ─────────────────────────
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32)

# ── Quantum device ─────────────────────────────────────
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

# ── Quantum Circuit (classifier) ───────────────────────
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):

    # Encode classical data
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")

    # Variational circuit
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    # Quantum classifier outputs (2 classes)
    return [qml.expval(qml.PauliZ(i)) for i in range(2)]

# trainable weights
weight_shapes = {"weights": (3, n_qubits, 3)}

qlayer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

# ── Wrapper to handle batches ──────────────────────────
class QuantumLayer(nn.Module):

    def __init__(self):
        super().__init__()
        self.qlayer = qlayer

    def forward(self, x):

        outputs = []

        for i in range(x.shape[0]):
            outputs.append(self.qlayer(x[i]))

        return torch.stack(outputs)

# ── Hybrid CNN + Quantum model ─────────────────────────
class HybridModel(nn.Module):

    def __init__(self):
        super().__init__()

        # CNN feature extractor
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(16, 32, 3)

        # compress features → qubits
        self.fc1 = nn.Linear(32*30*30, n_qubits)

        # quantum classifier
        self.quantum = QuantumLayer()

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = x.view(x.shape[0], -1)

        x = self.fc1(x)

        # normalize inputs for quantum circuit
        x = torch.tanh(x)

        # quantum classification
        x = self.quantum(x)

        return x

# ── Initialize model ───────────────────────────────────
model = HybridModel()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss()

# ── Training loop ──────────────────────────────────────
epochs = 10

for epoch in range(epochs):

    model.train()

    correct = 0
    total = 0
    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

# ── Testing accuracy ───────────────────────────────────
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total

print(f"\nTest Accuracy: {test_acc:.2f}%")

/tmp/ipykernel_540/141976079.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipykernel_540/141976079.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
/tmp/ipykernel_540/141976079.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
/tmp/ipykernel_540/141976079.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourc

Epoch 1/10 | Loss: 0.5881 | Train Acc: 74.45%
Epoch 2/10 | Loss: 0.5463 | Train Acc: 78.41%
Epoch 3/10 | Loss: 0.5269 | Train Acc: 79.27%
Epoch 4/10 | Loss: 0.5120 | Train Acc: 79.91%
Epoch 5/10 | Loss: 0.5033 | Train Acc: 80.09%
Epoch 6/10 | Loss: 0.4960 | Train Acc: 80.59%
Epoch 7/10 | Loss: 0.4885 | Train Acc: 81.14%
Epoch 8/10 | Loss: 0.4803 | Train Acc: 81.41%
Epoch 9/10 | Loss: 0.4747 | Train Acc: 82.32%
Epoch 10/10 | Loss: 0.4642 | Train Acc: 82.91%

Test Accuracy: 82.12%


In [ ]:
Rotation X
Epoch 1/10 | Loss: 0.6605 | Train Acc: 69.00%
Epoch 2/10 | Loss: 0.5933 | Train Acc: 77.95%
Epoch 3/10 | Loss: 0.5339 | Train Acc: 79.68%
Epoch 4/10 | Loss: 0.4902 | Train Acc: 80.32%
Epoch 5/10 | Loss: 0.4601 | Train Acc: 81.68%
Epoch 6/10 | Loss: 0.4407 | Train Acc: 82.59%
Epoch 7/10 | Loss: 0.4218 | Train Acc: 83.00%
Epoch 8/10 | Loss: 0.4060 | Train Acc: 83.64%
Epoch 9/10 | Loss: 0.3884 | Train Acc: 84.36%
Epoch 10/10 | Loss: 0.3752 | Train Acc: 84.77%

Test Accuracy: 83.18%
